# 02 — Heuristic Baseline

**Purpose:** Reproduce the SQL heuristic logic embedded in the
`gold_study_recommendations` Gold table in pure Python, then evaluate it with
the `scripts/metrics.py` evaluation framework.

This notebook establishes the **baseline performance** that all ML-based
recommenders must beat to be considered a real improvement.

## Heuristic Logic (reproduced from SQL)

For each `(user_id, course_id)` pair:

1. Find the **weakest concept** — the `node_id` with the highest `struggle_rate`
   for this user-course pair.
2. **If** a weakest concept exists (struggle data found):
   - `recommended_action_type = 'review_struggle_concept'`
3. **Else if** `check_accuracy < 0.60`:
   - `recommended_action_type = 'discuss_with_ai'`
4. **Else:**
   - `recommended_action_type = 'learn_next_lesson'`

## Expected Baseline Metrics (from DATA_ANALYST_RECOMMENDER_GUIDE.md)

| Metric | Baseline |
|--------|----------|
| Precision@5 | 0.40 |
| Recall@5 | 0.25 |
| nDCG@5 | 0.38 |

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
OUTPUT_DIR = '../output/evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Output dir:', OUTPUT_DIR)

## 1. Load Gold Tables

In [ ]:
def load_parquet_or_demo(table_name: str, gold_dir: str) -> pd.DataFrame:
    """Load a Gold Parquet table; fall back to synthetic demo data."""
    path = os.path.join(gold_dir, f'{table_name}.parquet')
    if os.path.exists(path):
        print(f'[Parquet] Loading {table_name} from {path}')
        return pd.read_parquet(path)

    print(f'[Demo] {table_name} not found. Generating synthetic data...')
    rng = np.random.default_rng(42)
    n_users, n_nodes, n_courses = 150, 60, 8
    user_ids = list(range(1, n_users + 1))
    node_ids = list(range(100, 100 + n_nodes))
    course_ids = list(range(1, n_courses + 1))

    if table_name == 'gold_concept_struggles':
        n = 1200
        return pd.DataFrame({
            'user_id':      rng.choice(user_ids, n),
            'node_id':      rng.choice(node_ids, n),
            'course_id':    rng.choice(course_ids, n),
            'struggle_rate': rng.uniform(0.0, 1.0, n).round(3),
            'attempt_count': rng.integers(1, 30, n),
        })
    else:  # gold_student_course_metrics
        n = 400
        return pd.DataFrame({
            'user_id':        rng.choice(user_ids, n),
            'course_id':      rng.choice(course_ids, n),
            'check_accuracy': rng.uniform(0.0, 1.0, n).round(3),
            'completion_rate': rng.uniform(0.0, 1.0, n).round(3),
            'total_attempts': rng.integers(1, 50, n),
        })


df_struggles  = load_parquet_or_demo('gold_concept_struggles',      GOLD_DIR)
df_metrics    = load_parquet_or_demo('gold_student_course_metrics',  GOLD_DIR)

print(f'gold_concept_struggles:      {df_struggles.shape}')
print(f'gold_student_course_metrics: {df_metrics.shape}')
display(df_struggles.head(3))
display(df_metrics.head(3))

## 2. Reproduce Heuristic Logic

Mirrors the SQL CASE WHEN logic used to build `gold_study_recommendations`.

In [ ]:
# Step 1: Find weakest concept per (user, course) — node with max struggle_rate
weakest_concept = (
    df_struggles
    .sort_values('struggle_rate', ascending=False)
    .drop_duplicates(subset=['user_id', 'course_id'], keep='first')
    [['user_id', 'course_id', 'node_id', 'struggle_rate']]
    .rename(columns={'node_id': 'weakest_node_id', 'struggle_rate': 'max_struggle_rate'})
)
print(f'Weakest concept pairs: {len(weakest_concept):,}')

# Step 2: Merge with course metrics to get check_accuracy
# Deduplicate course metrics to one row per (user, course)
df_metrics_dedup = (
    df_metrics
    .sort_values('total_attempts', ascending=False)
    .drop_duplicates(subset=['user_id', 'course_id'], keep='first')
)

heuristic_df = df_metrics_dedup.merge(
    weakest_concept, on=['user_id', 'course_id'], how='left'
)

# Step 3: Apply heuristic CASE WHEN logic
def apply_heuristic(row):
    if pd.notna(row.get('weakest_node_id')):
        return 'review_struggle_concept'
    elif row.get('check_accuracy', 1.0) < 0.60:
        return 'discuss_with_ai'
    else:
        return 'learn_next_lesson'

heuristic_df['recommended_action_type'] = heuristic_df.apply(apply_heuristic, axis=1)
heuristic_df['recommended_node_id'] = heuristic_df['weakest_node_id']

print('\nHeuristic output sample:')
display(heuristic_df[['user_id', 'course_id', 'check_accuracy',
                       'weakest_node_id', 'max_struggle_rate',
                       'recommended_action_type']].head(10))

## 3. Recommended Action Distribution

In [ ]:
action_dist = heuristic_df['recommended_action_type'].value_counts()
print('recommended_action_type distribution:')
print(action_dist.to_string())
print(f'\nTotal recommendations generated: {len(heuristic_df):,}')

## 4. Evaluation: Precision@5, Recall@5, nDCG@5

Build a ground truth dict and heuristic predictions dict, then evaluate using
`scripts/metrics.evaluate_model`.

In [ ]:
from scripts.metrics import evaluate_model, summarize_evaluation

# Build ground truth: user_id -> list of node_ids they actually interacted with
# We use gold_concept_struggles as a proxy for "nodes the user has engaged with"
ground_truth = (
    df_struggles
    .groupby('user_id')['node_id']
    .apply(list)
    .to_dict()
)

# Build heuristic predictions: user_id -> list of up to 5 recommended node_ids
# For users with review_struggle_concept, we take the top-5 highest struggle nodes
heuristic_predictions = {}

top5_struggles = (
    df_struggles
    .sort_values('struggle_rate', ascending=False)
    .groupby('user_id')['node_id']
    .apply(lambda x: x.head(5).tolist())
    .to_dict()
)

for user_id in ground_truth.keys():
    heuristic_predictions[user_id] = top5_struggles.get(user_id, [])

print(f'Users with ground truth: {len(ground_truth):,}')
print(f'Users with predictions:  {len(heuristic_predictions):,}')

# Evaluate
eval_df = evaluate_model(heuristic_predictions, ground_truth, k=5)
summary = summarize_evaluation(eval_df)

print('\nEvaluation summary (mean / std / median):')
display(summary)

## 5. Save Results

In [ ]:
# Save per-user evaluation results
out_path = os.path.join(OUTPUT_DIR, 'heuristic_baseline_metrics.csv')
eval_df.to_csv(out_path, index=False)
print(f'Per-user evaluation saved to: {out_path}')

# Save summary metrics
summary_path = os.path.join(OUTPUT_DIR, 'heuristic_baseline_summary.csv')
summary.to_csv(summary_path)
print(f'Summary metrics saved to:     {summary_path}')

# Compare against expected baseline from guide
print('\n=== Comparison with Guide Baseline ===')
guide_baseline = {'precision@5': 0.40, 'recall@5': 0.25, 'ndcg@5': 0.38}
for metric, expected in guide_baseline.items():
    # Normalize metric name in eval_df columns
    col = [c for c in eval_df.columns if metric.replace('@', '@') in c.lower()]
    if col:
        actual = eval_df[col[0]].mean()
        delta = actual - expected
        direction = '+' if delta >= 0 else ''
        print(f'  {metric:<15}: expected={expected:.2f}, actual={actual:.4f}, delta={direction}{delta:.4f}')
    else:
        print(f'  {metric:<15}: expected={expected:.2f}, not computed')